# Clasificación de compuertas lógicas dibujadas a mano

**PC2 · Computación Gráfica · UNI**

**Problema:** digitalizar circuitos lógicos dibujados a mano. Como primer paso, una red neuronal debe reconocer qué compuerta se dibujó.

**Clases (7):** AND, OR, NOT, NAND, NOR, XOR y **OTRO** (cualquier dibujo que no es una de esas 6 compuertas: letras, números, formas, flechas, garabatos). Sin esta clase, la red tiene que elegir una compuerta aunque le muestren un círculo o una letra.

**Datos:** dibujos recolectados por el equipo con la app web del repositorio (lienzo de 256×256). Cada PNG está en una carpeta con el nombre de su clase y el trazo va en el canal alfa. Los 500 dibujos de OTRO se generaron por programa (`generar_otro.py`) imitando el trazo del lienzo; su autor figura como `sintetico`.

**Contenido del notebook**
1. Carga de datos
2. Preprocesamiento
3. Exploración del dataset
4. División en entrenamiento, validación y prueba
5. Red neuronal convolucional (CNN)
6. Métricas: accuracy, precisión, recall, F1 y matriz de confusión
7. Guardar el modelo
8. Conclusiones

## 1. Carga de datos

El dataset final está en el repositorio como `dataset.zip`: 3500 dibujos, 500 por clase: 3000 compuertas dibujadas por el equipo en la app desplegada y 500 de OTRO generados con `generar_otro.py`. Si el archivo no está en la carpeta actual (por ejemplo, al abrir el notebook en Colab), se descarga del repositorio.

In [ ]:
import os, io, json, zipfile, urllib.request, collections
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

URL_DATASET = "https://github.com/Faalvial/compuertas_logicas/raw/main/dataset.zip"
RUTA_ZIP = "dataset.zip"
CARPETA = "dataset"
CLASES = ["AND", "OR", "NOT", "NAND", "NOR", "XOR", "OTRO"]

if not os.path.exists(RUTA_ZIP):
    print("Descargando", URL_DATASET)
    urllib.request.urlretrieve(URL_DATASET, RUTA_ZIP)

with zipfile.ZipFile(RUTA_ZIP) as zf:
    zf.extractall(CARPETA)
print("Dataset extraído en", CARPETA)

## 2. Preprocesamiento

Cada dibujo pasa por estos pasos:
1. **Canal alfa:** el lienzo es transparente y el trazo es opaco, así que el canal alfa ya es la imagen en escala de grises (0 = fondo, 255 = tinta).
2. **Recorte:** se recorta al rectángulo que contiene la tinta, con un margen. Así no importa en qué parte del lienzo ni de qué tamaño dibujó cada persona.
3. **Cuadrado:** se rellena con fondo para que quede cuadrado sin deformar la compuerta.
4. **Redimensionar:** a `TAM × TAM` píxeles (64 por defecto) y normalizar a valores entre 0 y 1.

In [ ]:
TAM = 64
MARGEN = 0.08

def recortar_cuadrado(ruta, margen=MARGEN):
    alfa = Image.open(ruta).convert("RGBA").getchannel("A")
    caja = alfa.getbbox()                     # rectángulo con tinta
    if caja is None:
        return None
    x0, y0, x1, y1 = caja
    lado = int(max(x1 - x0, y1 - y0) * (1 + 2 * margen))
    cuadrado = Image.new("L", (lado, lado), 0)
    cuadrado.paste(alfa.crop(caja), ((lado - (x1 - x0)) // 2, (lado - (y1 - y0)) // 2))
    return cuadrado

def a_matriz(recortes, tam):
    return np.stack([np.asarray(r.resize((tam, tam), Image.Resampling.LANCZOS), dtype=np.float32) / 255.0
                     for r in recortes])

recortes, etiquetas, autores, archivos = [], [], [], []
for i, clase in enumerate(CLASES):
    carpeta = os.path.join(CARPETA, clase)
    for nombre in sorted(os.listdir(carpeta)):
        if not nombre.endswith(".png"):
            continue
        r = recortar_cuadrado(os.path.join(carpeta, nombre))
        if r is None:
            continue
        recortes.append(r)
        etiquetas.append(i)
        partes = nombre.split("_")
        autores.append(partes[1] if len(partes) >= 3 else "anonimo")
        archivos.append(f"{clase}/{nombre}")

y = np.array(etiquetas)
autores = np.array(autores)
X = a_matriz(recortes, TAM)
print("Imágenes:", X.shape, " Etiquetas:", y.shape)

## 3. Exploración del dataset

In [ ]:
conteo = np.bincount(y, minlength=len(CLASES))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].bar(CLASES, conteo, color="#8a1c2b")
ax[0].set_title("Dibujos por clase")
for i, n in enumerate(conteo):
    ax[0].text(i, n, str(n), ha="center", va="bottom")
ax[0].set_ylim(0, conteo.max() * 1.15)
por_autor = collections.Counter(a for a in autores if a != "sintetico")  # solo dibujos del equipo
nombres = [a for a, _ in por_autor.most_common()]
ax[1].bar(nombres, [por_autor[a] for a in nombres], color="#5a626c")
ax[1].set_title("Compuertas dibujadas por persona")
for i, a in enumerate(nombres):
    ax[1].text(i, por_autor[a], f"{por_autor[a]} ({por_autor[a] / sum(por_autor.values()):.1%})", ha="center", va="bottom")
ax[1].set_ylim(0, max(por_autor.values()) * 1.15)
plt.tight_layout(); plt.show()
print(dict(zip(CLASES, conteo.tolist())), " total:", conteo.sum())

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(len(CLASES), 8, figsize=(10, 8))
for i, clase in enumerate(CLASES):
    idx = rng.choice(np.where(y == i)[0], size=min(8, (y == i).sum()), replace=False)
    for j in range(8):
        a = axes[i, j]; a.axis("off")
        if j < len(idx):
            a.imshow(X[idx[j]], cmap="gray_r")
    axes[i, 0].set_title(clase, loc="left", fontsize=10)
plt.suptitle(f"Ejemplos después del preprocesamiento ({TAM}×{TAM})"); plt.tight_layout(); plt.show()

La **imagen promedio** de cada clase muestra qué tan consistentes son los dibujos. En NAND y NOR debería verse el círculo de la salida; en XOR, la curva extra.

In [ ]:
fig, axes = plt.subplots(1, len(CLASES), figsize=(12, 2.4))
for i, clase in enumerate(CLASES):
    axes[i].imshow(X[y == i].mean(axis=0), cmap="magma"); axes[i].set_title(clase); axes[i].axis("off")
plt.suptitle("Imagen promedio por clase"); plt.tight_layout(); plt.show()

## 4. División en entrenamiento, validación y prueba

- **Entrenamiento (70%)**: la red aprende con estos datos.
- **Validación (15%)**: sirve para decidir cuándo dejar de entrenar.
- **Prueba (15%)**: solo se usa al final para medir el rendimiento real.

La división es **estratificada** (cada conjunto tiene la misma proporción de cada compuerta).

In [ ]:
from sklearn.model_selection import train_test_split
import logging
import keras
from keras import layers

logging.getLogger("tensorflow").setLevel(logging.ERROR)  # oculta avisos internos de TensorFlow

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)

indices = np.arange(len(y))
idx_train, idx_tmp = train_test_split(indices, test_size=0.30, stratify=y, random_state=SEMILLA)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, stratify=y[idx_tmp], random_state=SEMILLA)

def separar(Xs):
    Xs = Xs[..., None]  # canal de color
    return Xs[idx_train], Xs[idx_val], Xs[idx_test]

X_train, X_val, X_test = separar(X)
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]
print("Entrenamiento:", X_train.shape, " Validación:", X_val.shape, " Prueba:", X_test.shape)

## 5. Red neuronal convolucional (CNN)

La CNN aprende filtros que detectan bordes, curvas y círculos, y luego los combina. Eso la hace adecuada para distinguir detalles como el círculo de NAND o la curva extra de XOR.

**Aumento de datos:** durante el entrenamiento cada imagen se rota (±10°), se desplaza y se acerca o aleja un poco al azar, para simular la variación natural entre personas. **No** se aplica volteo horizontal: una compuerta volteada tendría la salida a la izquierda, que no es un dibujo válido.

**Arquitectura:** tres bloques de convolución + max pooling (32, 64 y 128 filtros), seguidos de una capa densa con dropout para reducir el sobreajuste.

**Entrenamiento:** hasta 20 épocas con `EarlyStopping`: si la pérdida de validación no mejora en 4 épocas seguidas, se detiene y se queda con los mejores pesos.

In [ ]:
EPOCAS = 20

def callbacks():
    return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)]

In [ ]:
def crear_aumento():
    return keras.Sequential([
        layers.RandomRotation(0.03, fill_mode="constant"),
        layers.RandomTranslation(0.08, 0.08, fill_mode="constant"),
        layers.RandomZoom(0.1, fill_mode="constant"),
    ], name="aumento_de_datos")

def crear_cnn(tam):
    return keras.Sequential([
        keras.Input((tam, tam, 1)),
        crear_aumento(),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dropout(0.4),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(len(CLASES), activation="softmax"),
    ], name="CNN")

cnn = crear_cnn(TAM)
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.summary()
hist_cnn = cnn.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCAS,
                   batch_size=64, callbacks=callbacks(), verbose=2)

### Curvas de entrenamiento

Si la pérdida de validación sube mientras la de entrenamiento baja, la red está memorizando (sobreajuste). `EarlyStopping` detiene el entrenamiento en ese punto y recupera los mejores pesos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(hist_cnn.history["loss"], label="entrenamiento")
axes[0].plot(hist_cnn.history["val_loss"], ls="--", label="validación")
axes[1].plot(hist_cnn.history["accuracy"], label="entrenamiento")
axes[1].plot(hist_cnn.history["val_accuracy"], ls="--", label="validación")
axes[0].set_title("Pérdida"); axes[1].set_title("Accuracy")
for a in axes:
    a.set_xlabel("Época"); a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Métricas en el conjunto de prueba

- **Accuracy:** porcentaje de dibujos clasificados correctamente.
- **Precisión** de una clase: de los dibujos que la red dijo que eran NAND, cuántos lo eran de verdad.
- **Recall** de una clase: de los NAND reales, cuántos encontró la red.
- **F1:** media armónica de precisión y recall.
- **Matriz de confusión:** fila = compuerta real, columna = compuerta predicha. Lo que está fuera de la diagonal son errores.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

prob_cnn = cnn.predict(X_test, verbose=0)
pred_cnn = prob_cnn.argmax(axis=1)
print(classification_report(y_test, pred_cnn, target_names=CLASES, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm = confusion_matrix(y_test, pred_cnn)
ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(ax=axes[0], cmap="Reds", colorbar=False)
axes[0].set_title("CNN: matriz de confusión (cantidades)")
cm_norm = confusion_matrix(y_test, pred_cnn, normalize="true")
ConfusionMatrixDisplay(cm_norm, display_labels=CLASES).plot(ax=axes[1], cmap="Reds", colorbar=False, values_format=".2f")
axes[1].set_title("CNN: matriz de confusión (por fila)")
for a in axes:
    a.set_xlabel("Predicha"); a.set_ylabel("Real")
plt.tight_layout(); plt.show()

### Dibujos que la CNN clasificó mal

In [ ]:
errores = np.where(pred_cnn != y_test)[0]
print(f"{len(errores)} errores de {len(y_test)} dibujos de prueba")
n = min(18, len(errores))
if n:
    cols = 6; filas = int(np.ceil(n / cols))
    fig, axes = plt.subplots(filas, cols, figsize=(12, 2.3 * filas), squeeze=False)
    for k, a in enumerate(axes.flat):
        a.axis("off")
        if k < n:
            i = errores[k]
            a.imshow(X_test[i, ..., 0], cmap="gray_r")
            a.set_title(f"real {CLASES[y_test[i]]}\npred {CLASES[pred_cnn[i]]} ({prob_cnn[i].max():.0%})", fontsize=9)
    plt.tight_layout(); plt.show()

## 7. Guardar el modelo

La app no usa TensorFlow, porque ocupa más de 1 GB y el plan gratuito de PythonAnywhere da 512 MB. Pero una red ya entrenada solo hace multiplicaciones con sus pesos, así que la app repite las mismas capas con numpy (archivo `prediccion.py`). Para eso basta guardar los pesos de las capas de convolución y densas en `pesos_cnn.npz`. El aumento de datos y el dropout no tienen pesos: solo actúan durante el entrenamiento.

Después se copia `pesos_cnn.npz` al repositorio, junto a `main.py`, y en PythonAnywhere se hace `git pull` y **Reload**. Si se cambia la arquitectura de la CNN, hay que cambiar también la función `predecir` de `prediccion.py`.

In [ ]:
cnn.save("modelo_compuertas.keras")
with open("clases.json", "w") as f:
    json.dump(CLASES, f)

pesos = [w for capa in cnn.layers if isinstance(capa, (layers.Conv2D, layers.Dense)) for w in capa.get_weights()]
np.savez("pesos_cnn.npz", *pesos)
print(f"Guardado modelo_compuertas.keras, clases.json y pesos_cnn.npz ({sum(w.size for w in pesos):,} pesos)")

try:
    from google.colab import files
    files.download("pesos_cnn.npz")
except ImportError:
    pass

## 8. Conclusiones

Completar los números después de ejecutar el notebook (marcados con **XX**).

1. **Resultado:** la CNN obtuvo **XX%** de accuracy en los 525 dibujos de prueba (**XX** errores), con F1 macro de **XX**. Las convoluciones reconocen formas locales (rectas, curvas, círculos) aunque no estén exactamente en el mismo lugar del dibujo.
2. **Por qué el resultado es tan alto:** son 7 formas muy distintas, dibujadas en el mismo lienzo, sin fondo ni ruido, y recortadas y centradas antes de entrar a la red. Además, la división es aleatoria, así que las mismas personas aparecen en entrenamiento y en prueba, y la red ya conoce su estilo.
3. **Errores:** las confusiones esperables son entre compuertas que solo se diferencian por un detalle: la burbuja (AND/NAND, OR/NOR) o la curva extra (OR/XOR). En la matriz de confusión se ve cuáles ocurrieron.
4. **La clase OTRO:** sin ella, la red tendría que elegir una compuerta para cualquier dibujo. Sus dibujos de prueba salen del mismo generador que los de entrenamiento, así que su resultado es optimista.
5. **Limitaciones:** las compuertas vienen de solo 4 personas, y el aumento de datos solo gira ±10°, así que una compuerta muy inclinada no se reconoce. Para digitalizar un circuito completo haría falta además detectar y separar cada compuerta y sus conexiones.